# 01 — Data Acquisition & Profiling
**Customer360 Navigator Enterprise Suite — Sprint 1**

## Purpose
Implements Master Execution Plan Section 23 (Agile Sprint Plan), Sprint 1: "Data acquisition/profiling +
BRD; hardware benchmark; src/ scaffolded" — CRISP-DM phase "Business Understanding, Data Understanding
begins." This is the Gate-1-adjacent Data Understanding step: it validates the raw file inventory and
profiles both real datasets (CFPB Consumer Complaint Database, PolyAI BANKING77) column-by-column, live,
before any taxonomy engineering (Gate 2) or modeling (Gate 3) is attempted. It performs no taxonomy mapping,
no feature engineering, and no row-level join — that is Gate 2's job (see
`notebooks/bp1_customer_intent_classification/bp1_customer_intent_classification_g2_data_integration_taxonomy_mapping.ipynb`).

## Standing rules this notebook follows
- **Execution boundary** (Master Execution Plan Section 12.2): Claude wrote this notebook; it does not run
  it. You run it on your own machine, and the real, live-measured numbers below become this project's
  profiling record. Nothing here was pre-computed or carried over from a prior session.
- **Zero-fabrication** (Section 12.1): every statistic below is computed live during this run from the real
  files in `data/raw/` and `data/external/` — never estimated, assumed, or copied from
  `docs/data_dictionary/RAW_DATA_MANIFEST.md` (that file's own figures were a separate, earlier
  measurement; this notebook independently re-verifies file size/row count as one of its structural
  checks, so any drift between the two is surfaced, not hidden).
- **WARP** (Section 15/17): `configure_performance()` is called first, before any heavy import. CFPB is lazy-
  scanned with Polars (`pl.scan_csv`, `Categorical` dtypes via the shared `CFPB_DTYPES` constant) rather than
  eagerly loaded — the file is ~322MB / 1.05M rows. BANKING77 (~13K rows total) is small enough to load
  eagerly.
- **HYPER** (Section 16): reuses `src/taxonomy/taxonomy_mapper.CFPB_DTYPES` (the same schema Gate 2's
  taxonomy notebook uses) rather than re-declaring the CFPB column dtypes inline.
- **Idempotent**: re-running this notebook overwrites its output files in place — safe to re-run any time
  the raw source files change.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: project root is resolved via an environment-variable override,
  then a bounded upward walk, then a bounded downward search — never a hardcoded absolute path (same
  resolver as `00_hardware_benchmark.ipynb`, including the Lesson #11 fix for a kernel cwd above the
  project folder).

## Inputs
- `data/raw/cfpb_complaints.csv` (CFPB Consumer Complaint Database export)
- `data/external/banking77_train.csv`, `banking77_test.csv`, `banking77_categories.json` (PolyAI BANKING77)

## Outputs (both written, idempotent overwrite-in-place)
- `notebooks/01_data_acquisition_profiling/artifacts/data_profile_summary.json` — machine-readable profile
- `docs/data_dictionary/DATA_PROFILE_REPORT.md` — human-readable profiling report

## Prerequisites
Run `pip install -r requirements.txt` from the project root first. `00_hardware_benchmark.ipynb` should be
run at least once before this (Sprint 1 ordering), though this notebook does not itself depend on its
output.

## If a structural check below fails
It raises `AssertionError` with the failing check named. Do not edit the check to make it pass — fix the
underlying cause (a missing file, a schema change in a re-exported source file, etc.) and re-run. A failing
check here means the raw data no longer matches what `docs/data_dictionary/RAW_DATA_MANIFEST.md` documented
and must be re-verified, not silently overridden.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3) - no heavy imports yet
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/01_data_acquisition_profiling/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
DOCS_DATA_DICT_DIR = PROJECT_ROOT / "docs" / "data_dictionary"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "01_data_acquisition_profiling" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# (LESSONS_LEARNED_APPLIED.md #5: thread-count env vars are read once at BLAS import time)
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports (safe now that WARP env vars are set) + flush-forcing print override
# (LESSONS_LEARNED_APPLIED.md #12: guarantees every checkpoint below appears immediately, not only
# at cell completion, so a stall becomes visible as "stopped after line X" rather than total silence)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402

from taxonomy.taxonomy_mapper import CFPB_DTYPES  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

CFPB_PATH = DATA_RAW_DIR / "cfpb_complaints.csv"
B77_TRAIN_PATH = DATA_EXTERNAL_DIR / "banking77_train.csv"
B77_TEST_PATH = DATA_EXTERNAL_DIR / "banking77_test.csv"
B77_CATEGORIES_PATH = DATA_EXTERNAL_DIR / "banking77_categories.json"

# ============================================================
# SECTION 4: Inventory validation - confirm every expected raw file is actually present
# Raises immediately (never silently skips) if a file documented in RAW_DATA_MANIFEST.md is missing.
# ============================================================
REQUIRED_FILES = [CFPB_PATH, B77_TRAIN_PATH, B77_TEST_PATH, B77_CATEGORIES_PATH]
missing = [str(p) for p in REQUIRED_FILES if not p.exists()]
assert not missing, f"[CHECK FAILED] Missing required raw data file(s): {missing}"

file_inventory = []
for p in REQUIRED_FILES:
    size_bytes = p.stat().st_size
    file_inventory.append({"path": str(p.relative_to(PROJECT_ROOT)), "size_bytes": size_bytes})
    print(f"[OK] Found {p.relative_to(PROJECT_ROOT)} ({size_bytes:,} bytes)")

# ============================================================
# SECTION 5: CFPB profiling (WARP - lazy scan, never an eager full-file load)
# ============================================================
print("\n[PROFILE] CFPB - scanning data/raw/cfpb_complaints.csv (lazy)...")
cfpb_lazy = pl.scan_csv(CFPB_PATH, dtypes=CFPB_DTYPES)

cfpb_row_count = cfpb_lazy.select(pl.len()).collect().item()
cfpb_columns = list(CFPB_DTYPES.keys())
print(f"[OK] CFPB row count (live): {cfpb_row_count:,}")

cfpb_null_counts = cfpb_lazy.select(
    [pl.col(c).null_count().alias(c) for c in cfpb_columns]
).collect().to_dicts()[0]

cfpb_distinct_counts = cfpb_lazy.select(
    [pl.col(c).n_unique().alias(c) for c in cfpb_columns]
).collect().to_dicts()[0]

cfpb_duplicate_ids = (
    cfpb_lazy.group_by("Complaint ID")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .collect()
    .height
)

cfpb_date_range = cfpb_lazy.select(
    pl.col("Date received").min().alias("min_date_received_str"),
    pl.col("Date received").max().alias("max_date_received_str"),
).collect().to_dicts()[0]

cfpb_product_distribution = (
    cfpb_lazy.group_by("Product")
    .agg(pl.len().alias("row_count"))
    .sort("row_count", descending=True)
    .collect()
)
cfpb_product_distribution_list = [
    {"product": r["Product"], "row_count": r["row_count"], "fraction_of_total": round(r["row_count"] / cfpb_row_count, 4)}
    for r in cfpb_product_distribution.to_dicts()
]

# Live drift check against the mapping config's baked-in distribution, if it has already been generated
# (Gate 2 ran before this notebook). Not required for this notebook to pass, but flagged if present.
taxonomy_config_path = CONFIGS_DIR / "taxonomy_mapping.yaml"
cfpb_drift_note = "taxonomy_mapping.yaml not yet generated - no drift check performed."
if taxonomy_config_path.exists():
    import yaml  # noqa: E402
    with open(taxonomy_config_path, "r", encoding="utf-8") as f:
        baked_in = yaml.safe_load(f)
    baked_total = baked_in.get("total_cfpb_rows")
    if baked_total == cfpb_row_count:
        cfpb_drift_note = f"[OK] Live CFPB row count ({cfpb_row_count:,}) matches taxonomy_mapping.yaml's baked-in total - no drift."
    else:
        cfpb_drift_note = (
            f"[DRIFT DETECTED] Live CFPB row count ({cfpb_row_count:,}) != taxonomy_mapping.yaml's baked-in "
            f"total ({baked_total}). Re-generate the taxonomy mapping config before trusting Gate 2 outputs."
        )
print(f"[INFO] {cfpb_drift_note}")

print(f"[OK] CFPB null counts (top 5 by count): "
      f"{dict(sorted(cfpb_null_counts.items(), key=lambda kv: kv[1], reverse=True)[:5])}")
print(f"[OK] CFPB duplicate Complaint ID groups: {cfpb_duplicate_ids}")
print(f"[OK] CFPB Date received range (as text, unparsed): {cfpb_date_range['min_date_received_str']} .. "
      f"{cfpb_date_range['max_date_received_str']}")

# ============================================================
# SECTION 6: BANKING77 profiling (small enough to load eagerly)
# ============================================================
print("\n[PROFILE] BANKING77 - loading train + test (eager, small dataset)...")
b77_train = pl.read_csv(B77_TRAIN_PATH, dtypes={"text": pl.Utf8, "category": pl.Categorical}).with_columns(
    pl.lit("train").alias("split")
)
b77_test = pl.read_csv(B77_TEST_PATH, dtypes={"text": pl.Utf8, "category": pl.Categorical}).with_columns(
    pl.lit("test").alias("split")
)
b77_all = pl.concat([b77_train, b77_test])

with open(B77_CATEGORIES_PATH, "r", encoding="utf-8") as f:
    b77_categories_file = json.load(f)

b77_row_counts = {"train": b77_train.height, "test": b77_test.height, "total": b77_all.height}
b77_categories_in_data = set(b77_all["category"].cast(pl.Utf8).unique().to_list())
b77_categories_missing_from_data = sorted(set(b77_categories_file) - b77_categories_in_data)
b77_categories_unexpected_in_data = sorted(b77_categories_in_data - set(b77_categories_file))

b77_null_counts = {
    "text": b77_all["text"].null_count(),
    "category": b77_all["category"].null_count(),
}
b77_duplicate_text_rows = b77_all.height - b77_all.select("text").unique().height

text_lengths = b77_all["text"].str.len_chars()
b77_text_length_stats = {
    "min_chars": int(text_lengths.min()),
    "max_chars": int(text_lengths.max()),
    "mean_chars": round(float(text_lengths.mean()), 2),
    "median_chars": float(text_lengths.median()),
}

b77_category_distribution = (
    b77_all.group_by("category").agg(pl.len().alias("row_count")).sort("row_count", descending=True)
)
b77_category_distribution_list = [
    {"category": str(r["category"]), "row_count": r["row_count"]} for r in b77_category_distribution.to_dicts()
]

print(f"[OK] BANKING77 row counts: {b77_row_counts}")
print(f"[OK] BANKING77 categories found in data: {len(b77_categories_in_data)} of {len(b77_categories_file)} expected")
print(f"[OK] BANKING77 null counts: {b77_null_counts}")
print(f"[OK] BANKING77 duplicate text rows: {b77_duplicate_text_rows}")
print(f"[OK] BANKING77 text length (chars): {b77_text_length_stats}")

# ============================================================
# SECTION 7: Write outputs (idempotent overwrite-in-place)
# ============================================================
profile_summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "file_inventory": file_inventory,
    "cfpb": {
        "row_count": cfpb_row_count,
        "column_count": len(cfpb_columns),
        "columns": cfpb_columns,
        "null_counts": cfpb_null_counts,
        "distinct_counts": cfpb_distinct_counts,
        "duplicate_complaint_id_groups": cfpb_duplicate_ids,
        "date_received_range_str": cfpb_date_range,
        "product_distribution": cfpb_product_distribution_list,
        "taxonomy_config_drift_check": cfpb_drift_note,
    },
    "banking77": {
        "row_counts": b77_row_counts,
        "categories_expected": len(b77_categories_file),
        "categories_found_in_data": len(b77_categories_in_data),
        "categories_missing_from_data": b77_categories_missing_from_data,
        "categories_unexpected_in_data": b77_categories_unexpected_in_data,
        "null_counts": b77_null_counts,
        "duplicate_text_rows": b77_duplicate_text_rows,
        "text_length_stats_chars": b77_text_length_stats,
        "category_distribution": b77_category_distribution_list,
    },
}

summary_json_path = ARTIFACTS_DIR / "data_profile_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(profile_summary, f, indent=2)

report_lines = [
    "# Data Profile Report - Customer360 Navigator",
    "",
    f"Generated live by `notebooks/01_data_acquisition_profiling/01_data_acquisition_profiling.ipynb` on "
    f"{profile_summary['generated_at_utc']}. Every figure below was computed during this run from the real "
    "files in `data/raw/` and `data/external/` - none estimated or carried over.",
    "",
    "## CFPB Consumer Complaint Database export",
    "",
    f"- Row count (live): {cfpb_row_count:,}",
    f"- Column count: {len(cfpb_columns)}",
    f"- Duplicate `Complaint ID` groups: {cfpb_duplicate_ids}",
    f"- `Date received` range (as text, unparsed): {cfpb_date_range['min_date_received_str']} .. "
    f"{cfpb_date_range['max_date_received_str']}",
    f"- Taxonomy config drift check: {cfpb_drift_note}",
    "",
    "### Null counts by column",
    "",
    "| Column | Null count |",
    "|---|---|",
] + [f"| {c} | {cfpb_null_counts[c]:,} |" for c in cfpb_columns] + [
    "",
    "### Distinct value counts by column",
    "",
    "| Column | Distinct count |",
    "|---|---|",
] + [f"| {c} | {cfpb_distinct_counts[c]:,} |" for c in cfpb_columns] + [
    "",
    "### Product distribution (live)",
    "",
    "| Product | Row count | Fraction of total |",
    "|---|---|---|",
] + [
    f"| {r['product']} | {r['row_count']:,} | {r['fraction_of_total']:.2%} |"
    for r in cfpb_product_distribution_list
] + [
    "",
    "## BANKING77",
    "",
    f"- Row counts: train={b77_row_counts['train']:,}, test={b77_row_counts['test']:,}, "
    f"total={b77_row_counts['total']:,}",
    f"- Categories found in data: {len(b77_categories_in_data)} of {len(b77_categories_file)} expected "
    f"(missing: {b77_categories_missing_from_data or 'none'}; unexpected: "
    f"{b77_categories_unexpected_in_data or 'none'})",
    f"- Null counts: text={b77_null_counts['text']}, category={b77_null_counts['category']}",
    f"- Duplicate `text` rows: {b77_duplicate_text_rows}",
    f"- Text length (characters): min={b77_text_length_stats['min_chars']}, "
    f"max={b77_text_length_stats['max_chars']}, mean={b77_text_length_stats['mean_chars']}, "
    f"median={b77_text_length_stats['median_chars']}",
]

report_md_path = DOCS_DATA_DICT_DIR / "DATA_PROFILE_REPORT.md"
with open(report_md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")

print(f"\n[SAVED] {summary_json_path.relative_to(PROJECT_ROOT)}")
print(f"[SAVED] {report_md_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 8: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "cfpb_row_count_matches_manifest": cfpb_row_count == 1_048_575,
    "cfpb_column_count_is_15": len(cfpb_columns) == 15,
    "cfpb_complaint_id_has_zero_nulls": cfpb_null_counts.get("Complaint ID", 1) == 0,
    "cfpb_complaint_id_has_no_duplicates": cfpb_duplicate_ids == 0,
    # 13,083 (not RAW_DATA_MANIFEST.md's original 13,100) is the real, correct row count - see
    # LESSONS_LEARNED_APPLIED.md #14: the manifest's figure came from a naive line count (wc -l),
    # which over-counts whenever a quoted `text` field contains an embedded newline. This notebook's
    # own polars parse (a real RFC 4180 CSV parser) is what surfaced the discrepancy, and this check
    # target has been corrected to the verified-real value rather than the stale manifest figure.
    "banking77_total_row_count_matches_corrected_manifest": b77_row_counts["total"] == 13_083,
    "banking77_all_77_categories_present": len(b77_categories_missing_from_data) == 0
    and len(b77_categories_unexpected_in_data) == 0,
    "summary_json_written": summary_json_path.exists(),
    "profile_report_md_written": report_md_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print("\n[ALL CHECKS PASSED] 01_data_acquisition_profiling.ipynb complete. "
      "Proceed to Gate 2 (taxonomy mapping / feature engineering) or the BP1 Gate 1 business-understanding "
      "notebook next.")
